# Survival Analysis and Mixed (Hierarchical) Models

Notebook for **Chapter 11** of the *stats* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/stats/ch11.html

Tip: change a number, predict the output, then run the cell again.

## 11.1 Time-to-Event Data and Censoring

In [ ]:
import numpy as np
t=np.array([2,3,3,5,6,6,8,9,12,12])
e=np.array([1,1,0,1,1,1,0,1,0,0])
print("mean naive all",t.mean(), "mean events only",t[e==1].mean())

## 11.2 The Kaplan-Meier Estimator

In [ ]:
S=1.0
print("time n_risk d  q  S")
for u in sorted(set(t[e==1])):
    n=(t>=u).sum(); d=((t==u)&(e==1)).sum()
    S*=1-d/n
    print(u,n,d,round(d/n,4),round(S,4))

In [ ]:
from lifelines import KaplanMeierFitter
kmf = KaplanMeierFitter()
kmf.fit(durations=t, event_observed=e)
print(kmf.median_survival_time_)

## 11.3 Hazard, a Simple Model, the Log-rank Test and Cox Regression

In [ ]:
from scipy.optimize import minimize
def nll(b):                          # negative log partial likelihood
    ll=0
    for i in np.where(E==1)[0]:
        R=T>=T[i]                    # risk set at this event time
        ll+=b*G[i]-np.log(np.exp(b*G[R]).sum())
    return -ll
r=minimize(lambda z:nll(z[0]),[0.0])
b=r.x[0]; print("beta",r.x,"HR",np.exp(r.x))
h=1e-4; se=1/np.sqrt((nll(b+h)-2*nll(b)+nll(b-h))/h**2)   # curvature of the loglik
print("se",se,"ci",np.exp(b-1.96*se),np.exp(b+1.96*se),"loglik",-r.fun,-nll(0.0))

In [ ]:
from lifelines import CoxPHFitter
cph = CoxPHFitter()
cph.fit(df, duration_col="time", event_col="event")   # df has time, event, plan_annual, ...
cph.print_summary()                                   # shows coef, exp(coef)=HR, CI, p
cph.check_assumptions(df)                             # tests proportional hazards

## 11.4 Grouped Data and the Idea of Partial Pooling

In [ ]:
from scipy.optimize import minimize
rng=np.random.default_rng(7)
ns=[3,5,5,8,10,20,40,60]
true=rng.normal(250,30,8)                      # true group means (tau = 30)
data={g:rng.normal(true[g],40,ns[g]) for g in range(8)}   # sigma = 40
def nll(p):                          # negative log-likelihood, random-intercept model
    mu,lt,ls=p; tau2=np.exp(lt); sig2=np.exp(ls); ll=0
    for v in data.values():
        v=np.array(v); n=len(v); ybar=v.mean(); ssq=((v-ybar)**2).sum()
        logdet=(n-1)*np.log(sig2)+np.log(sig2+n*tau2)
        quad=ssq/sig2+n*(ybar-mu)**2/(sig2+n*tau2)
        ll+=-0.5*(n*np.log(2*np.pi)+logdet+quad)
    return -ll
r=minimize(nll,[250,np.log(500),np.log(500)],method='Nelder-Mead')
mu,tau2,sig2=r.x[0],np.exp(r.x[1]),np.exp(r.x[2])
print("mu",mu,"tau",np.sqrt(tau2),"sig",np.sqrt(sig2),"icc",tau2/(tau2+sig2))
print("g n true_mean nopool pool w   err_nopool err_pool")
en=[];ep=[]
for g,v in data.items():
    w=tau2/(tau2+sig2/len(v)); pool=mu+w*(v.mean()-mu)
    en.append(v.mean()-true[g]); ep.append(pool-true[g])
    if g in (0,2,5,7): print(g,len(v),round(true[g],1),round(v.mean(),1),round(pool,1),round(w,3),round(abs(en[-1]),1),round(abs(ep[-1]),1))
print("(rows 1, 3, 4, 6 omitted here)")
print("rmse nopool",np.sqrt(np.mean(np.square(en))),"rmse pool",np.sqrt(np.mean(np.square(ep))))

In [ ]:
import statsmodels.formula.api as smf
m = smf.mixedlm("delivery_min ~ 1", df, groups=df["depot"]).fit()
print(m.summary())        # shows the fixed intercept, the group variance and the residual variance

## 11.5 Random Slopes and Going Further

A random intercept says "each depot has its own starting level". A **random slope** adds "each depot also reacts differently to order volume". Each group gets its own line, shrunk toward the average line: `y = (μ + a_g) + (β + b_g) × volume + ε`. In `statsmodels`: `smf.mixedlm("delivery_min ~ load", df, groups=df["depot"], re_formula="~load")` (not run here). Ignore for now: crossed random effects, non-normal outcomes (generalized linear mixed models, which join Chapter 8's GLMs with random effects), Bayesian hierarchical models (Chapter 9 priors do the shrinking), and survival "frailty" models. "Mixed", "multilevel", "hierarchical" and "random-effects" model all name the same family.

## 11.6 Mixed Models in Depth

Section 11.4 built the random-intercept model. Here we add the next three ideas that appear in real analyses: random slopes, the cost of ignoring clusters (the design effect), and cluster-robust standard errors as the cheap alternative. Everything is run with numpy and scipy, so the numbers are real; `statsmodels` is not installed here.

In [ ]:
import numpy as np
from scipy.optimize import minimize
rng = np.random.default_rng(31)
G, m = 30, 12                                    # 30 depots, 12 observations each
true_a = rng.normal(0, 20, G)                    # random intercept deviations (SD 20)
true_b = rng.normal(0, 3, G)                     # random slope deviations (SD 3)
vol = [rng.uniform(0, 10, m) for _ in range(G)]  # order volume
y = [200 + true_a[g] + (8 + true_b[g]) * vol[g] + rng.normal(0, 15, m) for g in range(G)]
def nll(p):
    b0, b1, l1, l2, z, ls = p
    s1, s2, sig2 = np.exp(l1), np.exp(l2), np.exp(2 * ls)
    r = np.tanh(z)
    D = np.array([[s1**2, r * s1 * s2], [r * s1 * s2, s2**2]])
    ll = 0
    for g in range(G):
        Z = np.column_stack([np.ones(m), vol[g]])
        V = Z @ D @ Z.T + sig2 * np.eye(m)
        res = y[g] - (b0 + b1 * vol[g])
        sign, logdet = np.linalg.slogdet(V)
        ll += -0.5 * (m * np.log(2 * np.pi) + logdet + res @ np.linalg.solve(V, res))
    return -ll
r = minimize(nll, [200, 8, np.log(15), np.log(2), 0.0, np.log(15)], method="Nelder-Mead",
             options={"maxiter": 4000, "xatol": 1e-4, "fatol": 1e-6})
b0, b1, l1, l2, z, ls = r.x
print("fixed intercept, slope :", round(b0, 1), round(b1, 2))
print("SD intercept, SD slope :", round(np.exp(l1), 1), round(np.exp(l2), 2), "(true 20 and 3)")
print("residual SD            :", round(np.exp(ls), 1), "(true 15)")
# shrunken (best linear unbiased) group lines vs separate OLS lines
s1, s2, sig2 = np.exp(l1), np.exp(l2), np.exp(2 * ls); rho = np.tanh(z)
D = np.array([[s1**2, rho * s1 * s2], [rho * s1 * s2, s2**2]])
raw, pooled = [], []
for g in range(G):
    Z = np.column_stack([np.ones(m), vol[g]])
    ols = np.linalg.lstsq(Z, y[g], rcond=None)[0]
    V = Z @ D @ Z.T + sig2 * np.eye(m)
    u = D @ Z.T @ np.linalg.solve(V, y[g] - Z @ np.array([b0, b1]))
    raw.append(ols[1]); pooled.append(b1 + u[1])
raw, pooled = np.array(raw), np.array(pooled)
true_slopes = 8 + true_b
print("slope RMSE, separate OLS lines :", round(np.sqrt(np.mean((raw - true_slopes) ** 2)), 2))
print("slope RMSE, partial pooling    :", round(np.sqrt(np.mean((pooled - true_slopes) ** 2)), 2))
print("spread of slopes: raw SD", round(raw.std(), 2), " shrunk SD", round(pooled.std(), 2))

fixed intercept, slope : 201.3 6.91
SD intercept, SD slope : 19.4 2.57 (true 20 and 3)
residual SD            : 16.6 (true 15)
slope RMSE, separate OLS lines : 1.97
slope RMSE, partial pooling    : 1.39
spread of slopes: raw SD 3.09  shrunk SD 2.18


In [ ]:
import numpy as np
rng = np.random.default_rng(5)
users, m = 20, 50                                  # 20 users, 50 page loads each
icc_true = 0.3
sd_between, sd_within = np.sqrt(0.3), np.sqrt(0.7) # total variance 1
ybar_runs = []
for _ in range(2000):                              # how variable is the overall mean, really?
    u = rng.normal(0, sd_between, users)
    y = u[:, None] + rng.normal(0, sd_within, (users, m))
    ybar_runs.append(y.mean())
true_se = np.std(ybar_runs)
u = rng.normal(0, sd_between, users); y = u[:, None] + rng.normal(0, sd_within, (users, m))
naive_se = y.std(ddof=1) / np.sqrt(users * m)      # pretends 1000 independent rows
cluster_se = y.mean(axis=1).std(ddof=1) / np.sqrt(users)   # treats each user as one unit
deff = 1 + (m - 1) * icc_true
print("true SD of the overall mean (2000 repeats):", round(true_se, 4))
print("naive SE (1000 rows independent)          :", round(naive_se, 4))
print("cluster SE (20 user means)                :", round(cluster_se, 4))
print("design effect 1+(m-1)ICC                  :", round(deff, 2))
print("effective sample size 1000/deff           :", round(users * m / deff, 1))
print("naive SE x sqrt(deff)                     :", round(naive_se * np.sqrt(deff), 4))

true SD of the overall mean (2000 repeats): 0.1259
naive SE (1000 rows independent)          : 0.0329
cluster SE (20 user means)                : 0.1391
design effect 1+(m-1)ICC                  : 15.7
effective sample size 1000/deff           : 63.7
naive SE x sqrt(deff)                     : 0.1304


## 11.7 Cox Regression in Practice, and Competing Risks

Section 11.3 fitted a Cox model on ten customers, where the interval was enormous. Here we use 600 simulated customers so the results mean something, then cover the three traps that decide whether a survival analysis is believed: non-proportional hazards, covariates that change over time, and more than one way to leave.

In [ ]:
import numpy as np
rng = np.random.default_rng(17)
n = 600
annual = (rng.random(n) < 0.4).astype(float)       # 1 = annual plan
tickets = rng.poisson(1.5, n).astype(float)         # support tickets in first month
# true model: hazard = 0.05 * exp(-0.7 * annual + 0.25 * tickets)
rate = 0.05 * np.exp(-0.7 * annual + 0.25 * tickets)
t_event = rng.exponential(1 / rate)
t_cens = rng.uniform(6, 36, n)                      # administrative / random censoring
time = np.minimum(t_event, t_cens); event = (t_event <= t_cens).astype(int)
X = np.column_stack([annual, tickets])
order = np.argsort(time); time, event, X = time[order], event[order], X[order]
def fit_cox(X, time, event, iters=25):
    beta = np.zeros(X.shape[1])
    for _ in range(iters):
        eta = np.exp(X @ beta)
        # risk-set sums via reverse cumulative sums (data sorted by time, no ties)
        S0 = np.cumsum(eta[::-1])[::-1]
        S1 = np.cumsum((eta[:, None] * X)[::-1], axis=0)[::-1]
        S2 = np.cumsum((eta[:, None, None] * X[:, :, None] * X[:, None, :])[::-1], axis=0)[::-1]
        ev = event == 1
        grad = (X[ev] - S1[ev] / S0[ev, None]).sum(axis=0)
        mean = S1[ev] / S0[ev, None]
        hess = (S2[ev] / S0[ev, None, None] - mean[:, :, None] * mean[:, None, :]).sum(axis=0)
        beta = beta + np.linalg.solve(hess, grad)
    return beta, np.sqrt(np.diag(np.linalg.inv(hess)))
beta, se = fit_cox(X, time, event)
print("events:", event.sum(), "of", n)
for name, b, s, tb in zip(["annual", "tickets"], beta, se, [-0.7, 0.25]):
    print(f"{name:8s} beta {b:6.3f} SE {s:5.3f} HR {np.exp(b):5.3f} 95% CI ({np.exp(b-1.96*s):.3f}, {np.exp(b+1.96*s):.3f})  true beta {tb}")

events: 388 of 600
annual   beta -0.737 SE 0.110 HR 0.479 95% CI (0.386, 0.594)  true beta -0.7
tickets  beta  0.179 SE 0.041 HR 1.196 95% CI (1.103, 1.296)  true beta 0.25


In [ ]:
import numpy as np
from scipy.optimize import minimize
from scipy.stats import chi2
rng = np.random.default_rng(23)
n = 800
x = (rng.random(n) < 0.5).astype(float)             # a promotion shown at sign-up
def simulate(fading):
    t = np.empty(n)
    for i in range(n):
        h1 = 0.08 * np.exp(-0.7 * x[i])             # promotion cuts the hazard by half ...
        if not fading:
            t[i] = rng.exponential(1 / h1)
        else:
            t1 = rng.exponential(1 / h1)            # ... but only for months 0-3
            t[i] = t1 if t1 < 3 else 3 + rng.exponential(1 / 0.08)
    c = rng.uniform(10, 40, n)
    return np.minimum(t, c), (t <= c).astype(int)
def ph_test(time, event, x):
    """Does the effect of x change with log(time)? Likelihood-ratio test of x*log(t) in a Cox model."""
    o = np.argsort(time); time, event, x = time[o], event[o], x[o]
    n1 = np.cumsum(x[::-1])[::-1]; n0 = np.cumsum((1 - x)[::-1])[::-1]   # risk-set counts, no ties
    ev = event == 1; g = np.log(time[ev])
    def nll(p, with_trend=True):
        theta = p[0] + (p[1] * g if with_trend else 0)
        return -(x[ev] * theta - np.log(n0[ev] + n1[ev] * np.exp(theta))).sum()
    r0 = minimize(lambda p: nll([p[0]], False), [0.0])
    r1 = minimize(nll, [0.0, 0.0])
    lr = 2 * (r0.fun - r1.fun)
    return np.exp(r0.x[0]), r1.x[1], lr, chi2.sf(lr, 1)
for fading in (False, True):
    time, event = simulate(fading)
    hr, slope, lr, p = ph_test(time, event, x)
    print("effect fades after month 3:" if fading else "constant effect (PH holds):",
          f"HR {hr:.2f}  log-HR change per unit log-time {slope:+.2f}  LR {lr:.2f}  p {p:.4f}")

constant effect (PH holds): HR 0.46  log-HR change per unit log-time +0.00  LR 0.00  p 0.9469
effect fades after month 3: HR 0.86  log-HR change per unit log-time +0.29  LR 16.26  p 0.0001


In [ ]:
import numpy as np
rng = np.random.default_rng(9)
n = 20000
lifetime = rng.exponential(10, n)                   # everyone: same hazard 0.1 per month, NO effect of upgrading
upgrade_at = rng.uniform(0, 24, n)                  # the month each customer WOULD upgrade if still around
upgraded = lifetime > upgrade_at                    # you can only upgrade while you are still subscribed
print("share who ever upgraded          :", round(upgraded.mean(), 3))
print("mean tenure, 'ever upgraded'      :", round(lifetime[upgraded].mean(), 2))
print("mean tenure, 'never upgraded'     :", round(lifetime[~upgraded].mean(), 2))
# correct: measure tenure AFTER the upgrade date (time at risk after joining the upgrade group)
after = (lifetime - upgrade_at)[upgraded]
print("mean time after upgrade point     :", round(after.mean(), 2), "(true mean 10)")

share who ever upgraded          : 0.376
mean tenure, 'ever upgraded'      : 17.74
mean tenure, 'never upgraded'     : 5.34
mean time after upgrade point     : 10.13 (true mean 10)


In [ ]:
import numpy as np
rng = np.random.default_rng(14)
n = 20000
lam1, lam2 = 0.05, 0.05                             # cancel (cause 1) and payment failure (cause 2), per month
t1 = rng.exponential(1 / lam1, n); t2 = rng.exponential(1 / lam2, n)
time = np.minimum(t1, t2); cause = np.where(t1 <= t2, 1, 2)
horizon = 24
true_cif = lam1 / (lam1 + lam2) * (1 - np.exp(-(lam1 + lam2) * horizon))
# naive: treat payment failures as censored and use 1 - Kaplan-Meier
ts = np.sort(np.unique(time[(cause == 1) & (time <= horizon)]))
S = 1.0
for u in ts:
    at_risk = (time >= u).sum(); d = ((time == u) & (cause == 1)).sum()
    S *= 1 - d / at_risk
naive = 1 - S
# correct: Aalen-Johansen cumulative incidence = sum of S_overall(t-) * d1 / n_at_risk
S_all, cif = 1.0, 0.0
for u in np.sort(np.unique(time[time <= horizon])):
    at_risk = (time >= u).sum()
    d1 = ((time == u) & (cause == 1)).sum(); d_all = (time == u).sum()
    cif += S_all * d1 / at_risk
    S_all *= 1 - d_all / at_risk
print("true cumulative incidence of cancel by month 24:", round(true_cif, 4))
print("naive 1 - KM (other cause censored)           :", round(naive, 4))
print("Aalen-Johansen estimate                        :", round(cif, 4))
print("share of customers still active at month 24    :", round((time > horizon).mean(), 4))

true cumulative incidence of cancel by month 24: 0.4546
naive 1 - KM (other cause censored)           : 0.7042
Aalen-Johansen estimate                        : 0.4592
share of customers still active at month 24    : 0.0894


## Chapter recap

## Practice set

Problem 11.P.1 A customer is still subscribed 9 months after sign-up. What do you record?

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/stats/ch11.html